# Predicting Anthocyanin Content from Colour Features: report notebook
**Mini project, Applied Analytics and AI (TX00GM35-3001), Mary Anne Selirio, October 2026**

This notebook reproduces **every number, table and figure in the final report**, in the same order as the report sections.
Each cell is labelled with the report location it feeds (e.g. *→ Report Table 4*).

| Notebook part | Report section |
|---|---|
| Part A · Load and check data | 3 Data and Preparation, Table 2 |
| Part B · Exploratory analysis | 4 Exploratory Analysis, Figures 1–3 |
| Part C · Workflow | 5 Method / Workflow, Figure 4, Table 3 |
| Part D · Main model (linear regression) | 6 Results, Table 4, Figure 5 |
| Part E · Diagnostics | 6 Results bullet points |
| Part F · Robustness check (beyond the course) | 6 Results, robustness paragraph, Figure 6 |
| Part G · KPI check | 6 Results, Table 5 |

Run all cells in order (Runtime → Run all in Google Colab). Random seed 42 everywhere, so results are identical on every run.
Data: Navarro et al. (2022), GigaDB doi:10.5524/102220 (CC0). Camera curves: Jiang et al. (2013), Zenodo 3245883 (CC BY-NC-SA 4.0).

In [ ]:
import os, warnings, urllib.request; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import (train_test_split, KFold, LeaveOneGroupOut, GroupKFold, GridSearchCV,
                                     cross_val_predict, cross_val_score, StratifiedKFold)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, silhouette_score, adjusted_rand_score

SEED = 42
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True,
                     "axes.titleweight": "bold"})
pd.set_option("display.precision", 3)

## Part A · Load and check the data (→ Report Section 3.1–3.3, Table 2)

In [ ]:
GH = "https://raw.githubusercontent.com/marysel-metropolia/anthocyanin-analysis/refs/heads/main/"
gt = pd.read_csv(GH + "groundTruth.csv")
sp = pd.read_csv(GH + "MSI_spectral-spatialCompressed.csv")
sp = sp.rename(columns={sp.columns[0]: "ArrayName"})
df = gt.merge(sp, on="ArrayName", validate="one_to_one")          # error if any berry ID repeats
df = df.rename(columns={"Anthocyanines.mg.kgFW.": "TAC", "Type": "variety", "Measure": "session"})
BANDS = [f"Channel{i}" for i in range(1, 38)]
ORDER = ["Itum4", "Itum5", "Crimson", "AutumRoyal", "Itum9"]
VCOL = dict(zip(ORDER, PAL))
y, day = df.TAC.values, df.session.values

print("Berries:", len(df), "| missing values:", int(df.isna().sum().sum()), "| sessions:", df.session.nunique())
print(df.variety.value_counts()[ORDER].to_dict())

**Simulated camera RGB** (→ Table 2): the 12 visible bands weighted by the Canon 1D Mark III sensitivity curves (Jiang et al., 2013), interpolated to the exact band wavelengths in Table 2 of Navarro et al. (2022). This is a *colorimetric proxy for spectral analysis*, not real phone photos.

In [ ]:
wl = np.array([488.38, 488.58, 503.59, 516.60, 530.62, 542.17, 567.96, 579.29, 592.89, 602.88, 616.59, 625.71])  # Navarro et al. 2022, Table 2
cam_wl = np.arange(400, 721, 10)
R = [0.0005826,0.0015345,0.0032648,0.0017968,0.0011327,0.0010138,0.0012358,0.0015814,0.0023156,0.0033231,0.0099827,0.019719,0.052282,0.11518,0.12991,0.10525,0.14334,0.28241,0.31153,0.55747,0.5377,0.49413,0.42527,0.36399,0.27563,0.21406,0.14996,0.068701,0.012535,0.0023522,0.00050291,0.00013838,4.4186e-05]
G = [0.0022632,0.0090549,0.040708,0.047983,0.069255,0.098384,0.11977,0.16981,0.31083,0.46114,0.94734,0.83935,0.89683,1,0.92794,0.87725,0.74845,0.63893,0.33715,0.32125,0.1676,0.082557,0.043512,0.029175,0.019359,0.013462,0.010043,0.0062015,0.0016749,0.00044468,0.00013311,3.9683e-05,1.7053e-05]
B = [0.010972,0.08325,0.41186,0.57035,0.71985,0.76168,0.8411,0.82581,0.78355,0.62952,0.68337,0.30674,0.14434,0.072663,0.041467,0.022853,0.010411,0.0064022,0.0032002,0.0032854,0.0022498,0.0014714,0.0012154,0.0011413,0.0013655,0.0017246,0.0019172,0.0011647,0.00023847,5.004e-05,1.6586e-05,9.5302e-06,8.494e-06]

S = np.vstack([np.interp(wl, cam_wl, c) for c in (R, G, B)])
def to_rgb(V, S): return V @ S.T / S.sum(axis=1)
df[["R", "G", "B"]] = to_rgb(df[BANDS[:12]].values, S)
RGB = ["R", "G", "B"]
df.groupby("variety")[RGB].mean().loc[ORDER].round(2)

## Part B · Exploratory analysis (→ Report Section 4)
### B1 · Distribution and group differences (→ Section 4.1)

In [ ]:
desc = df.groupby("variety").TAC.agg(["count", "mean", "median", "std"]).loc[ORDER]
print(desc.round(2))
print(f"\nAll berries: mean {df.TAC.mean():.1f}, median {df.TAC.median():.1f} mg/kg, skewness {df.TAC.skew():.1f}")
print(f"Shapiro–Wilk p = {stats.shapiro(df.TAC).pvalue:.1e}")
groups = [df.loc[df.variety == v, "TAC"] for v in ORDER]
print("Kruskal–Wallis H = %.0f, p = %.1e" % stats.kruskal(*groups))
print("Levene p = %.1e" % stats.levene(*groups).pvalue)

### B2 · Figure 1: anthocyanin per berry by variety

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i, v in enumerate(ORDER):
    vals = df.loc[df.variety == v, "TAC"].values
    jit = np.random.default_rng(SEED).uniform(-0.18, 0.18, len(vals))
    ax.scatter(np.full(len(vals), i) + jit, vals, s=9, color=VCOL[v], alpha=0.55, linewidths=0)
    ax.hlines(np.median(vals), i - 0.3, i + 0.3, color="k", lw=2)
ax.set_xticks(range(5), ORDER); ax.grid(axis="x", visible=False)
ax.set_ylabel("Anthocyanin (mg/kg fresh weight)")
ax.set_title("Pigment content per variety — three varieties sit near zero", loc="left")
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_tac_by_variety.png"); plt.show()

### B3 · Outliers and confounding (→ Sections 4.2, 4.3)

In [ ]:
def iqr_out(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum())
print("IQR outliers, pooled:", iqr_out(df.TAC), "| within each variety:", sum(iqr_out(g) for g in groups))
pd.crosstab(df.session, df.variety)[ORDER]      # each session holds one variety only

### B4 · Correlation, Figure 2, and multicollinearity (→ Sections 4.4, 4.5)

In [ ]:
r_band = df[BANDS].corrwith(df.TAC)
print(f"Strongest band: {r_band.abs().idxmax()} (626 nm), pooled r = {r_band[r_band.abs().idxmax()]:.2f}")
print("Within each variety:", {v: round(stats.pearsonr(df.loc[df.variety == v, 'Channel12'], df.loc[df.variety == v, 'TAC'])[0], 2) for v in ORDER})
print("p-values within variety:", {v: round(stats.pearsonr(df.loc[df.variety == v, 'Channel12'], df.loc[df.variety == v, 'TAC'])[1], 3) for v in ORDER})
print("Neighbouring bands, Channel1 vs Channel2: r =", round(df.Channel1.corr(df.Channel2), 3))

def vif(cols):
    X = sm.add_constant(df[cols].values)
    return np.array([variance_inflation_factor(X, i) for i in range(1, X.shape[1])])
v37, vrgb = vif(BANDS), vif(RGB)
print(f"VIF 37 bands: {v37.min():.0f} to {v37.max():.1e} | VIF R, G, B: {vrgb.min():.0f} to {vrgb.max():.0f}")

In [ ]:
# Figure 2: correlation of each band with anthocyanin (pooled). Band wavelengths from Navarro et al. (2022), Table 2.
WL = np.array([488.38, 488.58, 503.59, 516.60, 530.62, 542.17, 567.96, 579.29, 592.89, 602.88, 616.59, 625.71,
               676.25, 689.83, 714.87, 729.06, 741.80, 755.44, 767.48, 781.25, 792.57, 802.25, 823.75, 835.10,
               845.81, 856.52, 866.68, 876.49, 885.18, 894.26, 908.58, 916.63, 925.36, 931.99, 938.48, 946.14, 952.76])
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.axhline(0, color="#52514e", lw=0.8)
ax.plot(WL[:12], r_band.values[:12], color=PAL[0], lw=2, marker="o", ms=4, label="visible")
ax.plot(WL[12:], r_band.values[12:], color=PAL[1], lw=2, marker="o", ms=4, label="near-infrared")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Correlation with pigment (r)")
ax.set_title("Which wavelengths move with pigment", loc="left"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_band_correlation.png"); plt.show()

### B5 · Clusters and Figure 3: PCA (→ Section 4.6)
K-means on the standardised 37 bands; variety, session and TAC are **not** used.

In [ ]:
Z = StandardScaler().fit_transform(df[BANDS])
sil = {k: silhouette_score(Z, KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)) for k in range(2, 16)}
print("Silhouette range K = 2–15: %.2f to %.2f" % (min(sil.values()), max(sil.values())))
km13 = KMeans(13, n_init=10, random_state=SEED).fit_predict(Z)
print(f"K = 13: ARI with session = {adjusted_rand_score(day, km13):.2f} | ARI with variety = {adjusted_rand_score(df.variety, km13):.2f}")

pca = PCA(2, random_state=SEED).fit(Z); pcs = pca.transform(Z)
print(f"PC1 + PC2 explain {100 * pca.explained_variance_ratio_.sum():.0f} % of variance")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharey=True)
for ax, col, title in [(axes[0], "variety", "Coloured by variety"), (axes[1], "session", "Coloured by capture session")]:
    cats = ORDER if col == "variety" else sorted(df[col].unique())
    for i, c in enumerate(cats):
        m = (df[col] == c).values
        ax.scatter(pcs[m, 0], pcs[m, 1], s=8, alpha=0.7, linewidths=0, label=c,
                   color=VCOL[c] if col == "variety" else plt.cm.tab20.colors[i % 20])
    ax.set_title(title, loc="left"); ax.set_xlabel("PC1")
    ax.legend(frameon=False, fontsize=7, markerscale=2, ncol=1 if len(cats) <= 8 else 2)
axes[0].set_ylabel("PC2")
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_pca_variety_session.png"); plt.show()

## Part C · Figure 4: workflow (→ Report Section 5)

In [ ]:
from matplotlib.patches import FancyBboxPatch
steps=[("1  Ingest","groundTruth.csv +\nMSI spectra (GigaDB)\nmerge on ArrayName"),
("2  Validate","1,283 rows, 0 missing,\none-to-one join"),
("3  Prepare","37 bands (X_full)\nsimulated camera RGB\n(Canon 1D Mk III)"),
("4  Explore","descriptives, tests,\nr per band, PCA,\nK-means, VIF"),
("5  Model","linear regression\nvs mean baseline\n(check: PLS, RF, GB)"),
("6  Validate ×2","A: random 5-fold\nB: leave-one-\nsession-out"),
("7  Diagnose","within-purple R²,\nerror per day,\n28-camera sweep")]
fig,ax=plt.subplots(figsize=(15,2.9));ax.axis('off');w=1.9;g=0.25
for i,(t,d) in enumerate(steps):
    x=i*(w+g)
    c="#FF5600" if i==5 else "#3A3F44"
    ax.add_patch(FancyBboxPatch((x,0),w,0.55,boxstyle="round,pad=0.02,rounding_size=0.06",fc=c,ec="none"))
    ax.text(x+w/2,0.275,t,ha='center',va='center',color='white',fontsize=12,weight='bold')
    ax.text(x+w/2,-0.08,d,ha='center',va='top',fontsize=9,color="#222")
    if i<len(steps)-1: ax.annotate("",xy=(x+w+g-0.02,0.275),xytext=(x+w+0.02,0.275),arrowprops=dict(arrowstyle="-|>",color="#888",lw=1.5))
ax.set_xlim(-0.1,len(steps)*(w+g));ax.set_ylim(-0.62,0.6)
plt.savefig(f"{OUT}/fig0_workflow.png", dpi=200, bbox_inches="tight"); plt.show()

## Part D · Main model: multiple linear regression (→ Report Section 6, Table 4, Figure 5)
- **Test A · random:** random 5-fold (five 80/20 hold-outs); test berries share photo days with training berries.
- **Test B · new session:** each of the 13 sessions is held out once (leave-one-session-out).

In [ ]:
def metrics(yt, p): return {"R2": r2_score(yt, p), "MAE": mean_absolute_error(yt, p), "RMSE": mean_squared_error(yt, p) ** 0.5}
lin = lambda: make_pipeline(StandardScaler(), LinearRegression())
CV = {"A random": KFold(5, shuffle=True, random_state=SEED), "B new session": LeaveOneGroupOut()}

rows, preds = [], {}
for fname, cols in [("Camera RGB", RGB), ("37 bands", BANDS)]:
    for test, cv in CV.items():
        p = cross_val_predict(lin(), df[cols].values, y, cv=cv, groups=day)
        preds[(fname, test)] = p
        rows.append({"features": fname, "model": "Linear regression", "test": test, **metrics(y, p)})
for test, cv in CV.items():   # mean baseline: predict the training-set mean
    p = cross_val_predict(lin(), np.zeros((len(y), 1)), y, cv=cv, groups=day)
    rows.append({"features": "–", "model": "Mean baseline", "test": test, **metrics(y, p)})
table4 = pd.DataFrame(rows).pivot_table(index=["features", "model"], columns="test", values=["R2", "MAE"])
print("Table 4 (report shows R² for A and B, and MAE for B):"); table4.round(2)

In [ ]:
# Table 4 caption: single 20 % hold-out, the wine-exercise design
tr, te = train_test_split(np.arange(len(y)), test_size=0.2, random_state=SEED)
for fname, cols in [("RGB", RGB), ("37 bands", BANDS)]:
    m = lin().fit(df.loc[tr, cols], y[tr])
    print(f"20 % hold-out R², {fname}: {r2_score(y[te], m.predict(df.loc[te, cols])):.2f}")

In [ ]:
# Figure 5: RGB linear regression, predicted vs measured
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)
for ax, test in zip(axes, CV):
    p = preds[("Camera RGB", test)]
    for v in ORDER:
        k = (df.variety == v).values
        ax.scatter(y[k], p[k], s=8, alpha=.6, color=VCOL[v], label=v)
    ax.plot([0, 100], [0, 100], "k--", lw=1)
    ax.set_title(f"Test {test} · R² {r2_score(y, p):.2f}", loc="left", fontsize=11); ax.set_xlabel("Measured (mg/kg)")
axes[0].set_ylabel("Predicted (mg/kg)"); axes[0].legend(fontsize=8, markerscale=2)
fig.suptitle("Simulated camera RGB, linear regression: predicted vs measured", x=0.01, ha="left", fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_pred_vs_measured_linear.png"); plt.show()

## Part E · Diagnostics (→ Report Section 6, bullet points)

In [ ]:
pB = preds[("Camera RGB", "B new session")]
purple = df.variety.isin(["AutumRoyal", "Itum9"]).values
print(f"Purple berries: measured {y[purple].min():.0f}–{y[purple].max():.0f} mg/kg, "
      f"predicted {pB[purple].min():.0f}–{pB[purple].max():.0f} mg/kg (Test B)")
print("Green berries predicted below zero:", int((pB[~purple] < 0).sum()))

# Within purple varieties only (random 5-fold)
for fname, cols in [("RGB", RGB), ("37 bands", BANDS)]:
    p = cross_val_predict(lin(), df.loc[purple, cols].values, y[purple], cv=KFold(5, shuffle=True, random_state=SEED))
    print(f"Within purple R², {fname}: {r2_score(y[purple], p):.2f}")

In [ ]:
# New-session MAE by day (RGB)
per_day = (df.assign(err=np.abs(y - pB)).groupby(["session", "variety"])
             .agg(n=("TAC", "size"), mean_TAC=("TAC", "mean"), MAE=("err", "mean")).reset_index().sort_values("MAE"))
per_day.round(2)

In [ ]:
# Regression coefficients (standardised) vs correlation: blue sign reversal
Zr = (df[RGB] - df[RGB].mean()) / df[RGB].std()
ols = sm.OLS(df.TAC, sm.add_constant(Zr)).fit()
pd.DataFrame({"coef (mg/kg per SD)": ols.params[RGB], "r with TAC": df[RGB].corrwith(df.TAC)}).round(2)

## Part F · Robustness check, beyond the course (→ Report Section 6, "Robustness check", Figure 6)
Three more models with settings tuned **inside each training fold** (grouped by session), the same RGB from all 28 cameras, and a classifier that predicts the photo session. Only used to check that the conclusions do not depend on the model choice.

In [ ]:
inner = GroupKFold(4)
def models():
    return {"PLS": GridSearchCV(make_pipeline(StandardScaler(), PLSRegression()),
                                {"plsregression__n_components": [2, 3, 5, 8]}, cv=inner, scoring="r2"),
            "Random forest": GridSearchCV(RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
                                          {"min_samples_leaf": [3, 10]}, cv=inner, scoring="r2"),
            "Gradient boosting": GridSearchCV(HistGradientBoostingRegressor(random_state=SEED),
                                              {"max_depth": [3, None], "learning_rate": [0.05, 0.1]}, cv=inner, scoring="r2")}
rob = {}
for fname, cols in [("RGB", RGB), ("37 bands", BANDS)]:
    X = df[cols].values
    for test, cv in CV.items():
        for mname, m in models().items():
            p = np.zeros(len(y))
            for a, b in cv.split(X, y, day):
                m.fit(X[a], y[a], groups=day[a]); p[b] = np.ravel(m.predict(X[b]))
            rob[(fname, mname, test)] = r2_score(y, p)
print("R² (this cell takes a few minutes):"); pd.Series(rob).unstack().round(2)

In [ ]:
# Figure 6: all four models, random test vs new session
for fname, key in [("RGB", "Camera RGB"), ("37 bands", "37 bands")]:
    for test in CV: rob[(fname, "Linear", test)] = r2_score(y, preds[(key, test)])
mods = ["Linear", "PLS", "Random forest", "Gradient boosting"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, fname, title in [(axes[0], "37 bands", "37 bands"), (axes[1], "RGB", "Camera RGB (simulated)")]:
    xs = np.arange(len(mods))
    for k, (test, col) in enumerate([("A random", "#2a78d6"), ("B new session", "#eb6834")]):
        vals = [rob[(fname, m, test)] for m in mods]
        bars = ax.bar(xs + (k - 0.5) * 0.38, vals, 0.38, color=col, label="Random 5-fold" if k == 0 else "Leave-one-session-out")
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
    ax.axhline(0, color="k", lw=0.8); ax.set_xticks(xs, mods, fontsize=9); ax.set_title(title, loc="left")
axes[0].set_ylabel("R²"); axes[1].legend(frameon=False, fontsize=8)
fig.suptitle("Scores fall when the test session is new", x=0.01, ha="left", fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_model_comparison.png"); plt.show()

In [ ]:
# Can colour identify the photo session within one variety?
for v in ["Crimson", "Itum5", "AutumRoyal"]:
    s = df[df.variety == v]
    acc = cross_val_score(RandomForestClassifier(300, random_state=SEED), s[BANDS], s.session,
                          cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).mean()
    print(f"{v:10s} {s.session.nunique()} sessions: {acc:.0%} correct (chance {s.session.value_counts(normalize=True).max():.0%})")

In [ ]:
# RGB from all 28 cameras (PLS, 3 components, Test B). Needs camspec_database.txt in the repository.
path = "camspec_database.txt"
for url in [GH + path, "https://zenodo.org/records/3245883/files/camspec_database.txt?download=1"]:
    if os.path.exists(path): break
    try: urllib.request.urlretrieve(url, path)
    except Exception: pass
if os.path.exists(path):
    lines = [l for l in open(path).read().splitlines() if l.strip()]
    cams = {lines[i].strip(): [np.array(lines[i + k].split(), float) for k in (1, 2, 3)] for i in range(0, len(lines), 4)}
    sweep = {}
    for cname, curves in cams.items():
        Sc = np.vstack([np.interp(wl, cam_wl, c) for c in curves]); Xc = to_rgb(df[BANDS[:12]].values, Sc)
        p = cross_val_predict(make_pipeline(StandardScaler(), PLSRegression(3)), Xc, y, cv=LeaveOneGroupOut(), groups=day)
        sweep[cname] = r2_score(y, np.ravel(p))
    sweep = pd.Series(sweep).sort_values()
    print(f"{len(sweep)} cameras: R² {sweep.min():.2f} to {sweep.max():.2f}")
else:
    print("camspec_database.txt not found: upload it to the repository to run the camera sweep.")

## Part G · KPI check (→ Report Section 6, Table 5)

In [ ]:
r = lambda f, t: r2_score(y, preds[(f, t)])
rgbA, rgbB, b37A, b37B = r("Camera RGB", "A random"), r("Camera RGB", "B new session"), r("37 bands", "A random"), r("37 bands", "B new session")
mae_rgbB = mean_absolute_error(y, preds[("Camera RGB", "B new session")])
mae_base = mean_absolute_error(y, cross_val_predict(lin(), np.zeros((len(y), 1)), y, cv=LeaveOneGroupOut(), groups=day))
ari_s, ari_v = adjusted_rand_score(day, km13), adjusted_rand_score(df.variety, km13)
kpi = pd.DataFrame([
    ["O1 R² on new session", "≥ 0.50", f"{rgbB:.2f} (RGB); {b37B:.2f} (37 bands)", rgbB >= 0.50],
    ["O1 MAE vs baseline", f"< {mae_base:.2f} mg/kg", f"{mae_rgbB:.2f} (RGB)", mae_rgbB < mae_base],
    ["O2 RGB vs 37 bands (new session)", "≥ 70 % of 37-band R²", f"{rgbB:.2f} vs {b37B:.2f}", rgbB >= 0.7 * b37B],
    ["O3 R² drop A → B", "≤ 0.10", f"{rgbA - rgbB:.2f} (RGB); {b37A - b37B:.2f} (37 bands)", (rgbA - rgbB) <= 0.10],
    ["O3 Clusters follow variety", "ARI variety > ARI session", f"{ari_v:.2f} vs {ari_s:.2f}", ari_v > ari_s],
    ["O4 Reproducible", "Yes", "Runs from public URLs, seed 42", True]],
    columns=["KPI", "Target", "Result", "Met?"])
kpi["Met?"] = kpi["Met?"].map({True: "Yes", False: "No"})
print(f"KPIs met: {(kpi['Met?'] == 'Yes').sum()} of {len(kpi)}"); kpi

In [ ]:
# Export the main tables for the repository
table4.round(3).to_csv(f"{OUT}/table4_main_model.csv"); kpi.to_csv(f"{OUT}/table5_kpi.csv", index=False)
pd.Series(rob).unstack().round(3).to_csv(f"{OUT}/robustness_models.csv")
print("Saved to", OUT, ":", sorted(os.listdir(OUT)))